In [1]:
import sys
import os
import uuid
import json
from datetime import datetime, UTC

# 1. СИСТЕМНЕ ВЫРАВНИВАНИЕ ПУТЕЙ (Импорт ядра из core/)
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# Импортируем строгие Pydantic-модели и инфраструктуру ядра
try:
    from core.entities import NormQuery, NormDecision, NormGene
    print("✅ [Core] Модули ядра успешно импортированы.")
except ImportError as e:
    print(f"❌ [Ошибка Импорта] Не удалось загрузить core компоненты: {e}")

import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import matplotlib.pyplot as plt

# 2. ИНФРАСТРУКТУРНЫЙ СЛОЙ
class DashboardEventBus:
    def __init__(self, event_store=None):
        self._listeners = {}
        self.store = event_store

    def subscribe(self, event_type: str, callback):
        if event_type not in self._listeners: self._listeners[event_type] = []
        self._listeners[event_type].append(callback)

    def publish(self, event_type: str, event_data: dict):
        callbacks = self._listeners.get(event_type, []) + self._listeners.get("*", [])
        for callback in callbacks: callback(event_data)
        if self.store and hasattr(self.store, 'save'):
            self.store.save(event_data)

# 3. ИНТЕРФЕЙСНЫЙ ПУЛЬТ (Виджеты)
agent_dropdown = widgets.Dropdown(
    options=['Рада', 'Голова', 'Секретар'],
    value='Рада',
    description='👤 Суб\'єкт:',
    style={'description_width': 'initial'}
)

vote_toggle = widgets.ToggleButton(
    value=True,
    description='🗳️ Факт голосування: ТАК',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

quorum_toggle = widgets.ToggleButton(
    value=True,
    description='👥 Кворум: Є',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

run_button = widgets.Button(
    description='Запустити вивід MVP',
    button_style='info',
    icon='play',
    layout=widgets.Layout(width='410px', height='40px', margin='10px 0px 0px 0px')
)

output_area = widgets.Output()

# 4. ДИНАМИЧЕСКАЯ РЕАКТИВНОСТЬ ИНТЕРФЕЙСА
def on_vote_change(change):
    if change['new']:
        vote_toggle.description = '🗳️ Факт голосування: ТАК'
        vote_toggle.button_style = 'success'
    else:
        vote_toggle.description = '🗳️ Факт голосування: НІ'
        vote_toggle.button_style = 'danger'

def on_quorum_change(change):
    if change['new']:
        quorum_toggle.description = '👥 Кворум: Є'
        quorum_toggle.button_style = 'success'
    else:
        quorum_toggle.description = '👥 Кворум: НЕМАЄ'
        quorum_toggle.button_style = 'danger'

vote_toggle.observe(on_vote_change, names='value')
quorum_toggle.observe(on_quorum_change, names='value')

# 5. ОСНОВНОЙ ЦИКЛ СИМУЛЯЦИИ И ОТРИСОВКИ ГРАФА
def execute_inference_dashboard(b=None):
    with output_area:
        clear_output(wait=True)
        
        current_agent = agent_dropdown.value
        has_voted = vote_toggle.value
        has_quorum = quorum_toggle.value
        
        print(f"🕒 [Вхідна подія] {datetime.now(UTC).isoformat()}")
        print(f"📊 [Поточний Контекст] Суб'єкт: {current_agent} | Голос: {has_voted} | Кворум: {has_quorum}")
        print("=" * 70)
        
        trace = [{"desc": "Ініціалізація Тетради виводу."}]
        current_node = "START"
        
        if current_agent != "Рада":
            trace.append({"desc": f"❌ Порушення: Агент '{current_agent}' не має прав ухвалювати рішення за NORM-ACT-001."})
            result_modality = "Заборонено (PROH)"
            current_node = "AGENT_CHECK"
        elif not has_voted:
            trace.append({"desc": "❌ Порушення: Відсутній зафіксований факт голосування депутата."})
            result_modality = "Заборонено (PROH)"
            current_node = "FACT_VOTE"
        elif not has_quorum:
            trace.append({"desc": "❌ Порушення: На засіданні немає легітимного кворуму."})
            result_modality = "Заборонено (PROH)"
            current_node = "QUORUM_CHECK"
        else:
            trace.append({"desc": "✅ Успіх: Усі нормативні умови правила NORM-ACT-001 задоволено."})
            result_modality = "Дозволено (PERM)"
            current_node = "PERM"
            
        trace.append({"desc": "Обчислення нормативного переходу завершено."})
        
        print(f"⚖️ ВЕРДИКТ СИСТЕМИ: {result_modality}")
        print("\n📜 Трасування виводу (Trace):")
        for step in trace:
            print(f"  - {step['desc']}")
        print("=" * 70)
        
        G = nx.DiGraph()
        states = {
            "START": "Вхід", "AGENT_CHECK": "Суб'єкт", "FACT_VOTE": "Голосування",
            "QUORUM_CHECK": "Кворум", "PERM": "PERM", "PROH": "PROH"
        }
        G.add_nodes_from(states.keys())
        
        edges = [
            ("START", "AGENT_CHECK"), ("AGENT_CHECK", "FACT_VOTE"), ("AGENT_CHECK", "PROH"),
            ("FACT_VOTE", "QUORUM_CHECK"), ("FACT_VOTE", "PROH"), 
            ("QUORUM_CHECK", "PERM"), ("QUORUM_CHECK", "PROH")
        ]
        G.add_edges_from(edges)
        
        pos = {"START": (0, 2), "AGENT_CHECK": (2, 2), "FACT_VOTE": (4, 3), "QUORUM_CHECK": (6, 3), "PERM": (8, 4), "PROH": (5, 0)}
        
        node_colors = []
        for node in G.nodes():
            if node == current_node:
                node_colors.append("#2ECC71" if result_modality == "Дозволено (PERM)" else "#E74C3C")
            elif result_modality == "Заборонено (PROH)" and node == "PROH":
                node_colors.append("#E74C3C")
            else:
                node_colors.append("#BDC3C7")
                
        plt.figure(figsize=(9, 4.5), dpi=95)
        nx.draw_networkx_nodes(G, pos, node_size=1800, node_color=node_colors, alpha=0.9)
        nx.draw_networkx_labels(G, pos, labels=states, font_size=9, font_weight="bold")
        nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=18, edge_color="#7F8C8D", width=1.5)
        plt.axis('off')
        plt.show()

run_button.on_click(execute_inference_dashboard)

# 6. КОМПОНОВКА СЕТКИ ИНТЕРФЕЙСА (Исправлено: маргины вместо Spacer)
dashboard_ui = widgets.VBox([
    widgets.Label(value="⚙️ ПУЛЬТ КЕРУВАННЯ НОРМАТИВНИМ ДВИГУНОМ MVP (NKS-003)", style={'font_weight': 'bold'}),
    widgets.Box(layout=widgets.Layout(height='10px')), # Замена Spacer
    agent_dropdown,
    widgets.Box(layout=widgets.Layout(height='5px')),  # Замена Spacer
    widgets.HBox([vote_toggle, quorum_toggle]),
    run_button,
    widgets.Box(layout=widgets.Layout(height='15px')), # Замена Spacer
    output_area
], layout=widgets.Layout(padding='15px', border='1px solid #CCC', border_radius='5px', width='460px'))

# Финальный рендеринг дашборда на экран
display(dashboard_ui)
execute_inference_dashboard()

✅ [Core] Модули ядра успешно импортированы.
